In [ ]:
from the_well.data.normalization import ZScoreNormalization
from pathlib import Path
import numpy as np
import pickle
import fsspec
import torch
import yaml
import sys

PROJECT_ROOT = "/Users/gabrieleaccarino/Workspace/waveletflux"
sys.path.insert(0, PROJECT_ROOT)

from waveletflux.utils.well_data import get_well_dataset, get_well_inputs
from waveletflux.modules.wavelet_fm import WaveletFlowModule

print("Imports OK | MPS:", torch.backends.mps.is_available())
print(f"MPS allocated : {torch.mps.current_allocated_memory() / 1e9:.2f} GB")
print(f"MPS reserved  : {torch.mps.driver_allocated_memory()  / 1e9:.2f} GB")

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

In [ ]:
#CONFIG_PATH = f"{PROJECT_ROOT}/configs/turbulent_radiative_layer_2D/WaveletFlow_turbulent_radiative_layer_2D_3scales_db2_v0.0.toml"
well_base_path    = f"/Users/gabrieleaccarino/Workspace/multi-scale-wavelet-flow-matching/data/datasets"
well_dataset_name = 'turbulent_radiative_layer_2D'
stats_path        = f"{well_base_path}/{well_dataset_name}/stats.yaml"
ckpt_name         = "WaveletFlow_turbulent_radiative_layer_2D_3scales_db2_v1.0_20260323_093126-epoch=188-val_loss=0.0217_portable.ckpt"
#ckpt_name         = "WaveletFlow_turbulent_radiative_layer_2D_1scales_db2_v1.0_20260323_102317-epoch=178-val_loss=0.0096_portable.ckpt"
ckpt_base_name    = ckpt_name.split('_v')[0]
ckpt_path         = f"{PROJECT_ROOT}/experiments/turbulent_radiative_layer_2D/ckpts/{ckpt_name}"

In [ ]:
with fsspec.open(stats_path, mode='r') as f:
    stats = yaml.safe_load(f)

model = WaveletFlowModule.from_checkpoint(ckpt_path, map_location=device)
model.eval()
model.to(device)
print("Model loaded from checkpoint.")

In [ ]:
dataset = get_well_dataset(
    path    = well_base_path,
    physics = well_dataset_name,
    split   = 'test',
    steps   = -1,     # load the full trajectory
)

In [ ]:
N_TRAJ             = np.arange(len(dataset.metadata.n_steps_per_trajectory)) 
N_ROLLOUT_STEPS    = dataset.metadata.n_steps_per_trajectory[0]                         # autoregressive steps after the seed window
N_STEPS_INPUT      = 4                                                                  # Ti — sliding-window width (must match training)
NUM_SAMPLING_STEPS = 20                                                                 # ODE steps per prediction
N_ENSEMBLE         = 32                                                                 # flow-matching ensemble size
device_ids         = [device_idx for device_idx in range(torch.cuda.device_count())]    # GPU device indices for DataParallel

In [ ]:
pred_trajs = []
real_trajs = []

for traj_idx in N_TRAJ:
    print(f"\n{'━'*60}")
    print(f"  Trajectory {traj_idx}")
    print(f"{'━'*60}")

    item          = dataset[traj_idx]
    x_full, label = get_well_inputs(item, device=device)      # (N_ROLLOUT_STEPS, H, W, C), physical

    preds = model.rollout(
        seed          = x_full[:N_STEPS_INPUT].unsqueeze(0),  # (1, N_STEPS_INPUT, H, W, C), physical
        scalar_conds  = label.unsqueeze(0),                   # (1, num_cond), scalar conditions
        rollout_steps = N_ROLLOUT_STEPS,
        n_ensemble    = N_ENSEMBLE,
        verbose       = True,
    )
    # preds: (1, N_ENSEMBLE, N_ROLLOUT_STEPS, H, W, C)

    pred_trajs.append(preds.detach().cpu())
    real_trajs.append(x_full.unsqueeze(0).detach().cpu())


# Save to binary

In [ ]:
# save predicted trajectories to disk
save_dir = Path(f'{PROJECT_ROOT}/experiments/turbulent_radiative_layer_2D/results')
save_dir.mkdir(exist_ok=True)

pred_trajs_stacked = torch.cat(pred_trajs, dim=0)  # (N_TRAJ, N_ENSEMBLE, N_ROLLOUT_STEPS, H, W, C)
save_path = save_dir / f"{ckpt_base_name}_rollout_nsampling_steps{NUM_SAMPLING_STEPS}_nensemble{N_ENSEMBLE}_traj.pkl"

with open(save_path, "wb") as f:
    pickle.dump(pred_trajs_stacked, f)

# save real trajectories to disk, save only one time!!
#real_trajs_stacked = torch.cat(real_trajs, dim=0)  # (N_TRAJ, N_ROLLOUT_STEPS, H, W, C)
#save_path = save_dir / f"ground_truth_{well_dataset_name}_traj.pkl"

#with open(save_path, "wb") as f:
#    pickle.dump(real_trajs_stacked, f)